In [0]:
%sql
SHOW CATALOGS;

catalog
samples
system
workspace


In [0]:
%sql
show schemas in samples;

databaseName
accuweather
bakehouse
databricks
healthverity
information_schema
nyctaxi
sec
tpcds_sf1
tpcds_sf1000
tpch


In [0]:
%sql
show tables in samples.wanderbricks;

database,tableName,isTemporary
wanderbricks,amenities,false
wanderbricks,booking_updates,false
wanderbricks,bookings,false
wanderbricks,clickstream,false
wanderbricks,countries,false
wanderbricks,customer_support_logs,false
wanderbricks,destinations,false
wanderbricks,employees,false
wanderbricks,hosts,false
wanderbricks,page_views,false


In [0]:
tables = [
    "users",
    "properties",
    "hosts",
    "bookings",
    "payments",
    "booking_updates"
]

for t in tables:
    print("=" * 80)
    print(t)
    spark.table(f"samples.wanderbricks.{t}").printSchema()

users
root
 |-- user_id: long (nullable = false)
 |-- email: string (nullable = true)
 |-- name: string (nullable = true)
 |-- country: string (nullable = true)
 |-- user_type: string (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- is_business: boolean (nullable = true)
 |-- company_name: string (nullable = true)

properties
root
 |-- property_id: long (nullable = false)
 |-- host_id: long (nullable = true)
 |-- destination_id: long (nullable = true)
 |-- title: string (nullable = true)
 |-- description: string (nullable = true)
 |-- base_price: float (nullable = true)
 |-- property_type: string (nullable = true)
 |-- max_guests: integer (nullable = true)
 |-- bedrooms: integer (nullable = true)
 |-- bathrooms: integer (nullable = true)
 |-- property_latitude: float (nullable = true)
 |-- property_longitude: float (nullable = true)
 |-- created_at: date (nullable = true)

hosts
root
 |-- host_id: long (nullable = false)
 |-- name: string (nullable = true)
 |-- email

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.bronze;

In [0]:
%sql
show schemas in workspace;


databaseName
bronze
default
information_schema


In [0]:
%sql
create schema if not exists workspace.silver;

In [0]:
%sql
create schema if not exists workspace.gold;

In [0]:
%sql
create schema if not exists workspace.metadata;

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.raw;

In [0]:
%sql
CREATE TABLE workspace.metadata.metadata_config (
    table_name STRING,
    source_table STRING,
    target_table STRING,
    primary_key STRING,
    watermark_column STRING,
    load_type STRING,
    active_flag STRING
);

In [0]:
%sql
CREATE TABLE workspace.metadata.watermark_tracker (
    table_name STRING,
    last_watermark TIMESTAMP,
    current_watermark TIMESTAMP,
    last_run_status STRING,
    last_run_time TIMESTAMP
);

In [0]:
%sql
CREATE TABLE workspace.metadata.audit_log (
    run_id STRING,
    table_name STRING,
    rows_read BIGINT,
    rows_written BIGINT,
    rows_rejected BIGINT,
    status STRING,
    start_time TIMESTAMP,
    end_time TIMESTAMP
);

Populate metadata_config


In [0]:
%sql
INSERT INTO workspace.metadata.metadata_config VALUES
('users',
 'samples.wanderbricks.users',
 'workspace.bronze.users',
 'user_id',
 'created_at',
 'FULL',
 'Y');

num_affected_rows,num_inserted_rows
1,1


In [0]:
%sql
CREATE TABLE workspace.raw.users AS
SELECT *
FROM samples.wanderbricks.users;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT COUNT(*) FROM workspace.raw.users;

COUNT(*)
124509


In [0]:
%sql
UPDATE workspace.metadata.metadata_config
SET source_table = 'workspace.raw.users'
WHERE table_name = 'users';

num_affected_rows
1


In [0]:
%sql
SELECT * 
FROM workspace.metadata.watermark_tracker;

table_name,last_watermark,current_watermark,last_run_status,last_run_time


In [0]:
%sql
DESCRIBE workspace.metadata.watermark_tracker;

col_name,data_type,comment
table_name,string,null
last_watermark,timestamp,null
current_watermark,timestamp,null
last_run_status,string,null
last_run_time,timestamp,null


In [0]:
%sql
DESCRIBE workspace.metadata.metadata_config;

col_name,data_type,comment
table_name,string,null
source_table,string,null
target_table,string,null
primary_key,string,null
watermark_column,string,null
load_type,string,null
active_flag,string,null


In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.metadata.audit_log
(
    table_name STRING,
    layer STRING,
    load_type STRING,
    records_read BIGINT,
    records_written BIGINT,
    status STRING,
    error_message STRING,
    start_time TIMESTAMP,
    end_time TIMESTAMP,
    duration_seconds DOUBLE
)
USING DELTA;

In [0]:
%sql
SHOW TABLES IN workspace.raw;

database,tableName,isTemporary
raw,users,false


In [0]:
%sql
CREATE TABLE workspace.raw.properties AS
SELECT * FROM samples.wanderbricks.properties;

CREATE TABLE workspace.raw.hosts AS
SELECT * FROM samples.wanderbricks.hosts;

CREATE TABLE workspace.raw.bookings AS
SELECT * FROM samples.wanderbricks.bookings;

CREATE TABLE workspace.raw.payments AS
SELECT * FROM samples.wanderbricks.payments;

CREATE TABLE workspace.raw.booking_updates AS
SELECT * FROM samples.wanderbricks.booking_updates;

num_affected_rows,num_inserted_rows


In [0]:
%sql
describe table workspace.raw.properties;

col_name,data_type,comment
property_id,bigint,Unique identifier of this property
host_id,bigint,The identifier of the host of this property
destination_id,bigint,Foreign Key to 'destinations'
title,string,Title of the property listing
description,string,Description of the property
base_price,float,Base price for booking
property_type,string,"Type of property (e.g., house, apartment)"
max_guests,int,Maximum number of guests allowed
bedrooms,int,Number of bedrooms
bathrooms,int,Number of bathrooms


In [0]:
%sql
INSERT INTO workspace.metadata.metadata_config
VALUES
('properties',
'workspace.raw.properties',
'workspace.bronze.properties',
'property_id',
'created_at',
'INCREMENTAL',
'Y'),

('hosts',
'workspace.raw.hosts',
'workspace.bronze.hosts',
'host_id',
'joined_at',
'INCREMENTAL',
'Y'),

('bookings',
'workspace.raw.bookings',
'workspace.bronze.bookings',
'booking_id',
'updated_at',
'INCREMENTAL',
'Y'),

('payments',
'workspace.raw.payments',
'workspace.bronze.payments',
'payment_id',
'payment_date',
'INCREMENTAL',
'Y'),

('booking_updates',
'workspace.raw.booking_updates',
'workspace.bronze.booking_updates',
'booking_update_id',
'updated_at',
'INCREMENTAL',
'Y');

num_affected_rows,num_inserted_rows
5,5


In [0]:
%sql
UPDATE workspace.metadata.metadata_config
SET load_type = 'INCREMENTAL'
WHERE table_name = 'users';

num_affected_rows
1


In [0]:
%sql
ALTER TABLE workspace.metadata.metadata_config
ADD COLUMNS (
    scd_type STRING
);

In [0]:
%sql
UPDATE workspace.metadata.metadata_config
SET scd_type = 'SCD2'
WHERE target_table IN ('workspace.bronze.users', 'workspace.bronze.hosts');

num_affected_rows
2


In [0]:
%sql
UPDATE workspace.metadata.metadata_config
SET scd_type = 'SCD1'
WHERE target_table IN ('workspace.bronze.properties', 'workspace.bronze.bookings', 'workspace.bronze.payments', 'workspace.bronze.booking_updates');

num_affected_rows
4


In [0]:
%sql
SELECT
    target_table,
    load_type,
    scd_type
FROM workspace.metadata.metadata_config;

target_table,load_type,scd_type
workspace.bronze.users,INCREMENTAL,SCD2
workspace.bronze.hosts,INCREMENTAL,SCD2
workspace.bronze.properties,INCREMENTAL,SCD1
workspace.bronze.bookings,INCREMENTAL,SCD1
workspace.bronze.payments,INCREMENTAL,SCD1
workspace.bronze.booking_updates,INCREMENTAL,SCD1


In [0]:
%sql
describe table workspace.bronze.users;

col_name,data_type,comment
user_id,bigint,Unique identifier of the user
email,string,User's email address.
name,string,Name of the user.
country,string,Country of residence.
user_type,string,"Type of user (e.g., individual or business)."
created_at,timestamp,Account creation date.
is_business,boolean,Boolean indicating if the user is a business.
company_name,string,Name of the company (if user is a business).


In [0]:
%sql
describe table workspace.metadata.audit_log;

col_name,data_type,comment
run_id,string,null
table_name,string,null
layer,string,null
load_type,string,null
rows_read,bigint,null
rows_written,bigint,null
rows_rejected,bigint,null
status,string,null
error_message,string,null
start_time,timestamp,null
